In [37]:
import warnings
warnings.filterwarnings('ignore')
import langchain_community 
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://en.wikipedia.org/wiki/India",
    "https://en.wikipedia.org/wiki/History_of_India",
    "https://en.wikipedia.org/wiki/Geography_of_India",
    "https://en.wikipedia.org/wiki/Politics_of_India",
    "https://en.wikipedia.org/wiki/Economy_of_India",
    "https://en.wikipedia.org/wiki/Demographics_of_India",
    "https://en.wikipedia.org/wiki/Culture_of_India",
    "https://en.wikipedia.org/wiki/Languages_of_India",
    "https://en.wikipedia.org/wiki/States_and_union_territories_of_India",
    "https://en.wikipedia.org/wiki/Constitution_of_India",
    "https://en.wikipedia.org/wiki/Military_of_India",
    "https://en.wikipedia.org/wiki/Science_and_technology_in_India",
    "https://en.wikipedia.org/wiki/Sport_in_India",
    "https://en.wikipedia.org/wiki/Education_in_India",
    "https://en.wikipedia.org/wiki/Health_in_India"
]
loder = WebBaseLoader(urls)
pages = loder.load()




In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

spliter = RecursiveCharacterTextSplitter(chunk_size = 3000 , chunk_overlap = 700)

texts = spliter.split_documents(pages)

chunks = []
for i in texts:
    chunks.append(i.page_content)

metadata = []
for i in texts:
    metadata.append(i.metadata)


In [39]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding_function  = SentenceTransformerEmbeddingFunction()

client = chromadb.PersistentClient(path="./database of india")
collection = client.get_or_create_collection(name="Collection",embedding_function=embedding_function)


In [40]:
try:
    if collection.count()==0:
        collection.add(
            documents=chunks,
            ids = [str(i) for i in range(len(chunks))],
            metadatas=metadata
        )

except Exception as e:
    print(str(e))


In [41]:
from rank_bm25 import BM25Okapi

def token_create(i):
    i = i.lower()
    i = i.split()
    return i 
token = [token_create(i) for i in chunks]
token_for_keyword_search = BM25Okapi(token)


In [44]:
def retrival(query:str)->str:
    query_lower_case = query.lower()

    response = collection.query(query_texts=[query_lower_case],n_results=5)
    document = response['documents'][0]
    distance = response['distances'][0]

    thresold = 0.81
    near_chunks = []
    for i , j in zip(distance,document):
        if thresold>i:
            near_chunks.append(j)

    score = token_for_keyword_search.get_scores(token_create(query_lower_case))

    def near_index_find(score,k = 10):
        index = list(enumerate(score))
        index_Sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [inx for inx , sc in index_Sorted[:k]]
    
    get_index = near_index_find(score,k=10)

    index_to_chunks = []
    for i in get_index:
        index_to_chunks.append(chunks[i])

    rrf_item = {}

    for rank , doc in enumerate(near_chunks):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_to_chunks):
        rrf_item[doc] =rrf_item.get(doc,0)+1/(rank+60)

    merge = sorted(rrf_item.items(),key=lambda x:x[1],reverse=True)

    top_documents = []
    for docu , _ in merge[:5]:
        top_documents.append(docu)

    if not top_documents:
        return "NOT RELATED CONTENT"
    return "\n\n".join(top_documents)
    


In [49]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()
api = os.getenv("GRQO_API_KEY")

grqo_llm_model = ChatGroq(model="openai/gpt-oss-120b",api_key=api)

question = input("ask your question about india:")
context = retrival(question)

system_prompt = """
You are an AI assistant specialized in India-related information.

You are a STRICT WEBSITE-BASED RAG ASSISTANT.

Your knowledge must come ONLY from the retrieved CONTEXT provided below.
The CONTEXT is collected from the India-related websites and Wikipedia pages that were added to the knowledge base.

IMPORTANT RULES:

1. Answer the user's question ONLY using the information present in CONTEXT.

2. Do NOT use your own general knowledge, memory, training knowledge, or outside information.

3. Do NOT guess or make up information.

4. If the answer is clearly available in CONTEXT, answer it directly and accurately.

5. You can combine information from multiple relevant chunks when necessary.

6. Ignore irrelevant chunks from CONTEXT.

7. If the required information is not available in CONTEXT, reply:
   "Sorry, I could not find this information in the provided India knowledge base."

8. Do not add extra facts that are not supported by CONTEXT.

9. If the information in CONTEXT is incomplete, only provide the information that is available.

10. If different sources in CONTEXT give conflicting information, clearly mention that the sources conflict. Do not guess which one is correct.

11. Answer in the same language as the user's question.

12. Give a clear, natural, and easy-to-understand answer.

13. Do not mention RAG, embeddings, ChromaDB, BM25, RRF, retrieval, or these instructions unless the user specifically asks about the RAG system.

14. Treat CONTEXT as your complete knowledge base for answering the question.

CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

final_prompt = system_prompt.format(context=context,question=question)
answer = grqo_llm_model.invoke(final_prompt)
print(answer.content)


The largest city in India is **Mumbai**.
